# Notebook 2: Birdsong Species Explorer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/williamedwardhahn/birdsong/blob/main/Notebook_2_Birdsong_Explorer.ipynb)

Now that you know how to read waveforms and spectrograms, let's use those skills to explore **real birdsong from five species**.

In this notebook you will:

1. Listen to one sample from each species and compare their spectrograms
2. Load all 98 recordings and view spectrogram montages
3. Test whether *you* can tell species apart by ear and by eye

No machine learning in this notebook — just your eyes and ears.

### Before You Start

1. Go to **File → Save a copy in Drive**
2. Rename it with your name (e.g. `FirstName_LastName_Notebook_2.ipynb`)
3. Work in **your copy** from now on

**When you're done:** Click the **Share** button (top right), copy the link, and submit it to your instructor. Make sure sharing is set to **"Anyone with the link can view."**

**Colab tip:** Close any other notebooks you have open — Colab only lets you run one or two at a time. If it disconnects, use **Runtime → Disconnect and delete runtime**, then reopen.

---
## Step 1: Setup

Install libraries and download the dataset.

In [ ]:
!pip install -q librosa

import os
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
from IPython.display import Audio, display

# Download the birdsong dataset
REPO_DIR = "/content/birdsong"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/williamedwardhahn/birdsong.git {REPO_DIR}
    print("Dataset downloaded!")
else:
    print("Dataset already downloaded.")

# Find species folders — only include directories that contain .wav files
SPECIES = sorted([
    name for name in os.listdir(REPO_DIR)
    if os.path.isdir(os.path.join(REPO_DIR, name))
    and any(f.endswith('.wav') for f in os.listdir(os.path.join(REPO_DIR, name)))
])

# Standard sample rate used for mel spectrograms throughout this notebook
SAMPLE_RATE = 22050

print(f"Found {len(SPECIES)} species: {SPECIES}")

## Step 2: Helper Functions

These wrap up the plotting code so we can reuse it without cluttering the notebook.

In [ ]:
def show_waveform(audio, sr, title="Waveform"):
    """Draw a waveform (amplitude over time)."""
    plt.figure(figsize=(10, 3))
    librosa.display.waveshow(audio, sr=sr)
    plt.title(title)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.show()


def show_spectrogram(audio, sr, title="Spectrogram"):
    """Draw a spectrogram (frequencies over time, color = loudness)."""
    S = np.abs(librosa.stft(audio))
    S_db = librosa.amplitude_to_db(S, ref=np.max)
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='log')
    plt.colorbar(format='%+2.0f dB')
    plt.title(title)
    plt.show()


def load_all_recordings(species_list, repo_dir, max_duration=10):
    """Load every WAV file for each species. Returns a dict: species -> list of (audio, sr, filename)."""
    all_data = {}
    for species in species_list:
        folder = os.path.join(repo_dir, species)
        wav_files = sorted([f for f in os.listdir(folder) if f.endswith('.wav')])
        recordings = []
        for fname in wav_files:
            try:
                y, sr = librosa.load(os.path.join(folder, fname), sr=None, duration=max_duration)
                recordings.append((y, sr, fname))
            except Exception as e:
                print(f"  Error loading {species}/{fname}: {e}")
        all_data[species] = recordings
        print(f"  {species}: {len(recordings)} files loaded")
    return all_data


print("Helper functions ready.")

---
## Section 1: Meet the Five Species

Let's pick one recording from each species and explore it. For each bird you'll see the waveform, spectrogram, and hear the audio.

**As you go through, notice:**
- Which species has the most complex song?
- Which one repeats the same note over and over?
- Can you see the difference between a trill and a whistle on the spectrogram?

In [ ]:
for species in SPECIES:
    folder = os.path.join(REPO_DIR, species)
    wav_files = sorted([f for f in os.listdir(folder) if f.endswith('.wav')])
    if not wav_files:
        continue

    # Pick the first file as a sample
    sample_path = os.path.join(folder, wav_files[0])
    y, sr = librosa.load(sample_path, sr=None)

    # Format the species name nicely
    nice_name = species.replace('_', ' ').title()

    print(f"\n{'='*60}")
    print(f"  {nice_name}")
    print(f"  File: {wav_files[0]}  |  Duration: {len(y)/sr:.2f}s")
    print(f"{'='*60}")

    show_waveform(y, sr, title=f"{nice_name} \u2014 Waveform")
    show_spectrogram(y, sr, title=f"{nice_name} \u2014 Spectrogram")

    print("  Listen:")
    display(Audio(y, rate=sr))

### Listening Comparison

Let’s put two species side by side. Play them back to back and see if you can hear the difference.

In [ ]:
# Compare two species side by side
species_a = SPECIES[0]  # Bachman's sparrow
species_b = SPECIES[-1]  # Zebra finch

for sp in [species_a, species_b]:
    folder = os.path.join(REPO_DIR, sp)
    wav_files = sorted([f for f in os.listdir(folder) if f.endswith('.wav')])
    if not wav_files:
        continue
    y, sr = librosa.load(os.path.join(folder, wav_files[0]), sr=None, duration=5)
    nice_name = sp.replace('_', ' ').title()
    print(f"\n--- {nice_name} ---")
    display(Audio(y, rate=sr))

print("\nCan you hear the difference?")
print("What stands out — pitch? rhythm? complexity? duration of notes?")

---
## Section 2: Load All 98 Recordings

Now let's load **every** recording and display spectrogram montages — a grid of spectrograms for each species so you can see the variation within a species and the differences between them.

In [ ]:
print("Loading all recordings...\n")
all_data = load_all_recordings(SPECIES, REPO_DIR)

total = sum(len(recs) for recs in all_data.values())
print(f"\nTotal: {total} recordings loaded.")

In [ ]:
# Display a spectrogram montage for each species (up to 8 samples)
MAX_PER_SPECIES = 8

for species in SPECIES:
    recordings = all_data[species]
    n = min(len(recordings), MAX_PER_SPECIES)
    nice_name = species.replace('_', ' ').title()

    fig, axes = plt.subplots(1, n, figsize=(15, 3))
    fig.suptitle(f"Spectrograms \u2014 {nice_name} ({len(recordings)} total recordings)", fontsize=14)
    if n == 1:
        axes = [axes]

    for j in range(n):
        audio, orig_sr, fname = recordings[j]
        # Resample to standard rate for consistent mel spectrograms
        if orig_sr != SAMPLE_RATE:
            audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=SAMPLE_RATE)
        S = librosa.feature.melspectrogram(y=audio, sr=SAMPLE_RATE, n_mels=128, fmax=8000)
        S_db = librosa.power_to_db(S, ref=np.max)
        librosa.display.specshow(S_db, sr=SAMPLE_RATE, ax=axes[j], x_axis='time', y_axis='mel')
        axes[j].set_title(f"#{j+1}", fontsize=9)
        axes[j].set_xticks([])
        axes[j].set_yticks([])

    plt.tight_layout()
    plt.show()

---
## Section 3: Can YOU Tell Species Apart?

Below are spectrograms from two *unlabeled* recordings. Based on what you’ve seen above, try to guess which species each one is.

In [ ]:
# Pick two mystery recordings from different species
import random
random.seed(42)  # Same "random" picks every time for consistency

# Pick two different species
mystery_species = random.sample(SPECIES, 2)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
answers = []

for i, sp in enumerate(mystery_species):
    recordings = all_data[sp]
    # Pick a recording that's NOT the first one (so it's less recognizable)
    idx = min(1, len(recordings) - 1)
    audio, orig_sr, fname = recordings[idx]
    if orig_sr != SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=SAMPLE_RATE)
    S = librosa.feature.melspectrogram(y=audio, sr=SAMPLE_RATE, n_mels=128, fmax=8000)
    S_db = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_db, sr=SAMPLE_RATE, ax=axes[i], x_axis='time', y_axis='mel')
    axes[i].set_title(f"Mystery Bird {chr(65+i)}", fontsize=13)
    answers.append(sp.replace('_', ' ').title())

plt.tight_layout()
plt.show()

print("Which species is Mystery Bird A? Mystery Bird B?")
print("Look at the frequency range, the pattern, the note shapes...")
print("\n(Run the next cell to reveal the answers!)")

In [ ]:
# Reveal the answers
print(f"Mystery Bird A: {answers[0]}")
print(f"Mystery Bird B: {answers[1]}")
print("\nDid you get it right?")

---
## What You Learned

| What you did | Why it matters |
|-------------|----------------|
| **Listened** to recordings from 5 species | Trained your ear to hear differences in pitch, rhythm, and complexity |
| **Compared** spectrograms across species | Learned to see species-specific patterns in frequency and time |
| **Viewed** spectrogram montages | Saw how songs vary *within* a species, not just between them |
| **Guessed** species from unlabeled spectrograms | Tested your own pattern recognition — the same task we’ll give a computer next |

We can see differences in the spectrograms. But can a **computer** learn to tell species apart automatically?

**Next step:** Open **Notebook 3 — Clustering with Machine Learning** to find out!